In [5]:
!pip install google-search-results

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=3f9746c072b6b8b67d5c2649f3661541d1281017bd94fad18e818316468a0647
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


In [ ]:
!pip install tavily-python

In [ ]:
import google.generativeai as genai
from serpapi import GoogleSearch
from google.colab import userdata

GEMINI_KEY = userdata.get('GEMINI_KEY')
SERP_KEY   = userdata.get('SERP_KEY')
TAVILY_API_KEY = userdata.get('TAVILY_API_KEY')

genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')

# Set SEARCH_PROVIDER to 'serpapi' or 'tavily' to choose the search backend
SEARCH_PROVIDER = 'serpapi'

print("Keys loaded! Model ready.")

In [7]:
def search_web(query: str) -> str:
    """Search Google and return top 3 results as text."""
    params = {
        "engine": "google",
        "q": query,
        "api_key": SERP_KEY,
        "num": 3
    }
    results = GoogleSearch(params).get_dict()
    organic = results.get("organic_results", [])

    if not organic:
        return "No results found."

    output = ""
    for i, r in enumerate(organic[:3], 1):
        title   = r.get("title", "")
        snippet = r.get("snippet", "")
        link    = r.get("link", "")
        output += f"{i}. {title}\n{snippet}\nURL: {link}\n\n"

    return output.strip()

print(search_web("What is agentic AI?"))

1. What is Agentic AI?
Agentic AI is an artificial intelligence system that can accomplish a specific goal with limited supervision. It consists of ai agents—machine learning models that mimic human decision-making to solve problems in real time.
URL: https://www.ibm.com/think/topics/agentic-ai


In [ ]:
from tavily import TavilyClient

def search_web_tavily(query: str) -> str:
    """Search the web using Tavily and return top 3 results as text."""
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(
        query=query,
        search_depth="basic",
        max_results=3
    )
    results = response.get("results", [])

    if not results:
        return "No results found."

    output = ""
    for i, r in enumerate(results[:3], 1):
        title   = r.get("title", "")
        snippet = r.get("content", "")
        link    = r.get("url", "")
        output += f"{i}. {title}\n{snippet}\nURL: {link}\n\n"

    return output.strip()


def search_web_dispatch(query: str) -> str:
    """Route search to the selected provider based on SEARCH_PROVIDER."""
    if SEARCH_PROVIDER == 'tavily':
        return search_web_tavily(query)
    return search_web(query)

print("Tavily search and dispatch ready!")

In [ ]:
def run_agent(user_question: str) -> str:
    print(f"\nQuestion: {user_question}\n" + "-"*50)

    system_prompt = """You are a helpful research agent.
You have access to a web search tool. To search, reply with:
SEARCH: your search query here

After getting results, analyze them and give a final answer.
When you have enough info, reply with:
ANSWER: your final answer here

Always search at least once before answering."""

    messages = [
        {"role": "user", "parts": [system_prompt]},
        {"role": "model", "parts": ["Understood! I'll search before answering."]},
        {"role": "user", "parts": [user_question]}
    ]

    for step in range(5):
        response = model.generate_content(messages)
        reply = response.text.strip()
        print(f"Agent: {reply}\n")

        if reply.startswith("SEARCH:"):
            query = reply[7:].strip()
            print(f"Searching for: {query}")
            results = search_web_dispatch(query)
            print(f"Results received...\n")
            messages.append({"role": "model", "parts": [reply]})
            messages.append({"role": "user",  "parts": [f"Search results:\n{results}"]})

        elif reply.startswith("ANSWER:"):
            return reply[7:].strip()

        else:
            messages.append({"role": "model", "parts": [reply]})

    return "Agent reached max steps without a final answer."

print("Agent function ready!")

In [10]:
answer = run_agent("What are the states in india?")
print("\n" + "="*50)
print("FINAL ANSWER:")
print("="*50)
print(answer)


Question: What are the states in india?
--------------------------------------------------
Agent: SEARCH: states in India

Searching for: states in India
Results received...

Agent: ANSWER: India is a federal union that comprises 28 states.


FINAL ANSWER:
India is a federal union that comprises 28 states.
